# Training The Model

## 1. Settin Up Path for CSV FIle and GDrive

In [6]:
import os
import pandas as pd

#CSV path of TOday's  words DF_1_to_25
CSV_PATH = r"G:\My Drive\Cleaned_Csv_Files\cleaned_df_1_to_25.csv"

if os.path.exists(CSV_PATH):
    df = pd.read_csv(CSV_PATH)
    
    # Extract unique words (glosses) into a clean list
    new_words = df['gloss'].unique().tolist()
    
    print(f"Successfully loaded {len(new_words)} unique words for today's training.")
    print(f"Target Words: {new_words}")
    
    # Quick sanity check
    if len(new_words) != 25:
        print(f"⚠️ Heads up: Expected 25 words, but found {len(new_words)}.")
else:
    print(f"ERROR: Could not find the CSV file at {CSV_PATH}. Double-check the path!")

Successfully loaded 25 unique words for today's training.
Target Words: ['accident', 'africa', 'all', 'apple', 'basketball', 'bed', 'before', 'bird', 'birthday', 'black', 'blue', 'book', 'bowling', 'brown', 'but', 'can', 'candy', 'chair', 'change', 'cheat', 'city', 'clothes', 'color', 'computer', 'cook']


## 2. To Set up path For ASL_Videos, Aug_Data

In [7]:
import numpy as np
import pickle
import os


BASE_PATH = 'G:/My Drive/' # Suggested root folder
BASE_PATH2 = r'C:\Users\Uma_Manikanta\3D Objects\asl-translator\datasets'
DATA_PATH = os.path.join(BASE_PATH, 'ASL_Keypoints_Data')
AUG_DATA_PATH = os.path.join(BASE_PATH2, 'ASL_Augmented_Data')
LABEL_MAP_PATH = os.path.join(BASE_PATH, 'actions.pkl')

# Create folder if it don't exist
os.makedirs(AUG_DATA_PATH, exist_ok=True)

# 2. Load or Initialize the Label Map (The "Master List" of words)
if os.path.exists(LABEL_MAP_PATH):
    with open(LABEL_MAP_PATH, 'rb') as f:
        actions_list = list(pickle.load(f))
    print(f"Loaded existing Label Map with {len(actions_list)} words.")
else:
    actions_list = []
    print("No existing Label Map found. Starting fresh.")

# 3. Add TODAY'S new words to the Master List if they aren't already there
# 'new_words' comes from the previous cell (your CSV)
added_count = 0
for word in new_words:
    if word not in actions_list:
        actions_list.append(word)
        added_count += 1

# 4. Save the updated Master List back to your Drive
actions = np.array(actions_list)
with open(LABEL_MAP_PATH, 'wb') as f:
    pickle.dump(actions, f)

# 5. Create the mapping dictionary for the model
label_map = {label: num for num, label in enumerate(actions)}

print(f"Update complete. Added {added_count} new words.")
print(f"Total vocabulary size: {len(actions)}")

Loaded existing Label Map with 25 words.
Update complete. Added 0 new words.
Total vocabulary size: 25


# 3. To Augmenting the Data and Saving

In [8]:
import os
import numpy as np

# 1. Define the Augmentation Function tailored for daily updates
def apply_augmentation(source_path, target_path, words_to_augment, n_generated=5):
    print("Starting augmentation process...")
    
    for action in words_to_augment:
        action_path = os.path.join(source_path, action)
        save_action_path = os.path.join(target_path, action)
        
        # Check if raw data actually exists for this word
        if not os.path.exists(action_path):
            print(f"⚠️ Warning: No raw data found for '{action}'. Skipping.")
            continue
            
        os.makedirs(save_action_path, exist_ok=True)
        
        # CRITICAL FIX: Skip if we already augmented this word to save time
        if len(os.listdir(save_action_path)) > 0:
            print(f"⏩ Skipping '{action}': Already augmented.")
            continue

        video_files = [f for f in os.listdir(action_path) if f.endswith('.npy')]
        print(f"⚙️ Augmenting word: '{action}' ({len(video_files)} original files)...")

        for video_file in video_files:
            # Load original sequence
            res = np.load(os.path.join(action_path, video_file))
            
            # Save original as variation 0
            np.save(os.path.join(save_action_path, f"orig_{video_file}"), res)

            # Generate synthetic variations
            for i in range(n_generated):
                # Gaussian Noise (Simulates camera grain/hand jitter)
                noise = np.random.normal(0, 0.005, res.shape)
                augmented_res = res + noise

                # Spatial Shift (Simulates user moving slightly in frame)
                shift = np.random.uniform(-0.02, 0.02)
                augmented_res = augmented_res + shift

                # Save the augmented version
                np.save(os.path.join(save_action_path, f"aug_{i}_{video_file}"), augmented_res)

# 2. Execute ONLY for today's new words
apply_augmentation(DATA_PATH, AUG_DATA_PATH, new_words)
print("\n✅ Augmentation complete for today's batch!")

Starting augmentation process...
⚙️ Augmenting word: 'accident' (8 original files)...
⚙️ Augmenting word: 'africa' (10 original files)...
⚙️ Augmenting word: 'all' (10 original files)...
⚙️ Augmenting word: 'apple' (12 original files)...
⚙️ Augmenting word: 'basketball' (14 original files)...
⚙️ Augmenting word: 'bed' (13 original files)...
⚙️ Augmenting word: 'before' (11 original files)...
⚙️ Augmenting word: 'bird' (14 original files)...
⚙️ Augmenting word: 'birthday' (9 original files)...
⚙️ Augmenting word: 'black' (14 original files)...
⚙️ Augmenting word: 'blue' (13 original files)...
⚙️ Augmenting word: 'book' (13 original files)...
⚙️ Augmenting word: 'bowling' (14 original files)...
⚙️ Augmenting word: 'brown' (13 original files)...
⚙️ Augmenting word: 'but' (9 original files)...
⚙️ Augmenting word: 'can' (12 original files)...
⚙️ Augmenting word: 'candy' (1 original files)...
⚙️ Augmenting word: 'chair' (12 original files)...
⚙️ Augmenting word: 'change' (11 original files).

In [9]:
from sklearn.model_selection import train_test_split
from tensorflow.keras.utils import to_categorical

# 1. We already have 'actions' and 'label_map' from Cell 2 in memory.
# Let's double-check the label map just to be sure.
print(f"Total classes to load: {len(label_map)}")

# 2. Load sequences (X) and labels (y)
sequences, labels = [], []

# We iterate through ALL words in our Master List, not just 'new_words'
print("Loading all augmented data into memory... (This might take a moment as your dataset grows)")

for action in actions: # 'actions' is our master list of all words
    action_folder = os.path.join(AUG_DATA_PATH, action)
    
    # Skip if the folder doesn't exist yet for some reason
    if not os.path.exists(action_folder):
        continue
        
    files = [f for f in os.listdir(action_folder) if f.endswith('.npy')]

    for file in files:
        res = np.load(os.path.join(action_folder, file))
        sequences.append(res)
        labels.append(label_map[action])

# 3. Convert to Numpy arrays
X = np.array(sequences)
# One-hot encode the labels using the TOTAL number of actions we know about
y = to_categorical(labels, num_classes=len(actions)).astype(int)

# 4. Split the data
# Using 10% for testing. If you want a stricter evaluation for your 95%+ goal, 
# you could bump this to 0.20 (20%), but 10% is fine while building up data.
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.10, random_state=42)

print(f"\n✅ Data successfully loaded and split!")
print(f"X_train shape: {X_train.shape}") # Should be (Samples, 30 frames, 1662 keypoints)
print(f"y_train shape: {y_train.shape}") # Should be (Samples, Number of Words)

Total classes to load: 25
Loading all augmented data into memory... (This might take a moment as your dataset grows)

✅ Data successfully loaded and split!
X_train shape: (1474, 25, 1662)
y_train shape: (1474, 25)
